# Crime Count Forecasting — LSTM

## PART 1 — Training Pipeline (2021–2023)

> ⚠️ The 2024 dataset is **not loaded anywhere in this section**.
> The LSTM is trained on weekly sequences built from 2021–2023 only.
> The scaler and model are saved to disk. Part 2 loads 2024 as a fully fresh holdout.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("Imports complete.")
print(f"TensorFlow version: {tf.__version__}")


### Step 1 · Load Training Data (2021–2023 Only)

In [ ]:
def to_csv_url(link):
    m = re.search(r"/d/([A-Za-z0-9_-]+)", link)
    if not m:
        raise ValueError(f"Not a valid Drive file URL: {link}")
    return f"https://drive.google.com/uc?export=download&id={m.group(1)}"

TRAIN_LINKS = [
    "https://drive.google.com/file/d/1rgFQUYCOhxjyfcCK90oQCrYn7uVEn8Fn/view?usp=sharing",  # 2021
    "https://drive.google.com/file/d/1aJ_cmeHrXQzutYtAxb-wGqJV75z8jv8u/view?usp=sharing",  # 2022
    "https://drive.google.com/file/d/1C1iAJmsXjx-Ux-xC77EQ3-cOUO5Kkiru/view?usp=sharing",  # 2023
]

train_raw_df = pd.concat(
    [pd.read_csv(to_csv_url(link)) for link in TRAIN_LINKS],
    ignore_index=True
)
print(f"Training raw shape (2021-2023): {train_raw_df.shape}")


### Step 2 · Clean Training Data

In [ ]:
COLUMNS_TO_DROP = [
    'X', 'Y', 'CCN', 'REPORT_DAT', 'BLOCK',
    'XBLOCK', 'YBLOCK', 'BLOCK_GROUP', 'CENSUS_TRACT',
    'VOTING_PRECINCT', 'BID', 'END_DATE', 'OBJECTID', 'OCTO_RECORD_ID'
]
CRITICAL_COLS = ['START_DATE', 'LATITUDE', 'LONGITUDE', 'DISTRICT', 'PSA', 'NEIGHBORHOOD_CLUSTER']

def clean_df(df, label, date_start, date_end):
    df = df.copy()
    drop_cols = [c for c in COLUMNS_TO_DROP if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)

    before = len(df)
    df.dropna(subset=[c for c in CRITICAL_COLS if c in df.columns], inplace=True)
    print(f"[{label}] Dropped {before - len(df)} rows with missing critical cols")

    df['START_DATE'] = pd.to_datetime(df['START_DATE'], errors='coerce', utc=True)
    df.dropna(subset=['START_DATE'], inplace=True)

    start_ts = pd.Timestamp(date_start, tz='UTC')
    end_ts   = pd.Timestamp(date_end,   tz='UTC')
    before = len(df)
    df = df[(df['START_DATE'] >= start_ts) & (df['START_DATE'] <= end_ts)]
    print(f"[{label}] Dropped {before - len(df)} rows outside date range")

    if 'WARD' in df.columns:
        before = len(df)
        df.dropna(subset=['WARD'], inplace=True)
        print(f"[{label}] Dropped {before - len(df)} rows with missing WARD")

    print(f"[{label}] Final clean shape: {df.shape}\n")
    return df.reset_index(drop=True)

train_df = clean_df(train_raw_df, 'TRAIN', '2021-01-01', '2023-12-31 23:59:59')


### Step 3 · Time Features

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['year']         = df['START_DATE'].dt.year
    df['week_of_year'] = df['START_DATE'].dt.isocalendar().week.astype(int)
    return df

train_df = add_time_features(train_df)
print(f"Year range: {train_df['year'].min()} – {train_df['year'].max()}")


### Step 4 · Weekly Aggregation

Aggregate crime counts to `(year, week_of_year, NEIGHBORHOOD_CLUSTER)`. **Cluster set and parquet saved to disk** for use in Part 2.

In [ ]:
def build_weekly_agg(df, label, all_clusters=None):
    weekly_counts = (
        df.groupby(['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER'])
          .size()
          .reset_index(name='crime_count')
    )
    clusters = all_clusters if all_clusters is not None else df['NEIGHBORHOOD_CLUSTER'].unique()
    years    = range(df['year'].min(), df['year'].max() + 1)
    weeks    = range(1, 54)

    full_index = pd.MultiIndex.from_product(
        [years, weeks, clusters],
        names=['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER']
    )
    weekly_agg = (
        weekly_counts
        .set_index(['year', 'week_of_year', 'NEIGHBORHOOD_CLUSTER'])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .sort_values(['NEIGHBORHOOD_CLUSTER', 'year', 'week_of_year'])
        .reset_index(drop=True)
    )
    print(f"[{label}] Weekly aggregated shape: {weekly_agg.shape}")
    return weekly_agg

# Save cluster set so Part 2 uses the same 46 clusters
TRAIN_CLUSTERS = train_df['NEIGHBORHOOD_CLUSTER'].unique()
joblib.dump(TRAIN_CLUSTERS, 'lstm_train_clusters.pkl')
print(f"Saved {len(TRAIN_CLUSTERS)} clusters to lstm_train_clusters.pkl")

train_weekly = build_weekly_agg(train_df, 'TRAIN', all_clusters=TRAIN_CLUSTERS)

# Save the training weekly parquet for reference
train_weekly.to_parquet('lstm_train_weekly.parquet')
print("Training weekly parquet saved to lstm_train_weekly.parquet")
print(train_weekly.head())


### Step 5 · Scale Data

MinMaxScaler fitted on **2021–2023 training data only**. Saved to disk so Part 2 applies the identical scaling to 2024.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_weekly[['crime_count']])

train_weekly['scaled_count'] = scaler.transform(train_weekly[['crime_count']])

# Save scaler — Part 2 MUST use this, not refit on 2024
joblib.dump(scaler, 'lstm_scaler.pkl')
print("Scaler fitted on 2021-2023 and saved to lstm_scaler.pkl")
print(f"Scale range: [{scaler.data_min_[0]:.1f}, {scaler.data_max_[0]:.1f}] crimes")


### Step 6 · Build LSTM Sequences (Training)

Sliding window of `N_PAST_WEEKS=4` → predict next week's scaled count. Sequences built **per cluster** to avoid cross-cluster contamination.

In [ ]:
N_PAST_WEEKS = 4

def create_sequences(scaled_values, n_past):
    """Sliding window: [t-n, ..., t-1] → t"""    X, y = [], []
    for i in range(n_past, len(scaled_values)):
        X.append(scaled_values[i - n_past:i])
        y.append(scaled_values[i])
    return np.array(X), np.array(y)

all_X_train, all_y_train = [], []

for cluster in TRAIN_CLUSTERS:
    cluster_scaled = train_weekly[
        train_weekly['NEIGHBORHOOD_CLUSTER'] == cluster
    ]['scaled_count'].values

    if len(cluster_scaled) >= N_PAST_WEEKS + 1:
        X_c, y_c = create_sequences(cluster_scaled, N_PAST_WEEKS)
        all_X_train.append(X_c)
        all_y_train.append(y_c)

X_train_lstm = np.concatenate(all_X_train, axis=0)
y_train_lstm = np.concatenate(all_y_train, axis=0)

# Reshape for LSTM: [samples, timesteps, features]
X_train_lstm = X_train_lstm.reshape((X_train_lstm.shape[0], N_PAST_WEEKS, 1))

print(f"X_train_lstm shape: {X_train_lstm.shape}")
print(f"y_train_lstm shape: {y_train_lstm.shape}")

# Save N_PAST_WEEKS so Part 2 uses the same window
joblib.dump(N_PAST_WEEKS, 'lstm_n_past_weeks.pkl')


### Step 7 · Define and Train LSTM

Trained on 2021–2023 sequences only. Model saved to disk — **no test data seen yet**.

In [ ]:
print("--- Building LSTM Model ---")

model = Sequential(name="LSTM_Crime_Forecaster")
model.add(LSTM(
    units=50,
    activation='relu',
    input_shape=(N_PAST_WEEKS, 1)
))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation='linear'))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\n--- Training LSTM (2021-2023) ---")
# Use 10% of training data as internal validation during training
val_split = 0.1
history = model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=50,
    batch_size=32,
    validation_split=val_split,
    callbacks=[early_stopping],
    verbose=1
)

# Save model
model.save('lstm_model.keras')
print("\nLSTM trained and saved to lstm_model.keras")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Training Loss (2021-2023)')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.tight_layout()
plt.show()


---

# ═══════════════════════════════════════════════
# HOLDOUT TEST — 2024 DATA INTRODUCED HERE
# ═══════════════════════════════════════════════

> The cells above are **complete and frozen**.
> Nothing below existed during training.
> The scaler and model are loaded from disk — no refitting, no peeking.

**2024 Pipeline:**
1. Load raw 2024 CSV (link 4) fresh
2. Apply identical cleaning and time features
3. Weekly aggregation using saved cluster set
4. Scale using **saved scaler** (not refitted)
5. Build sequences — drop first `N_PAST_WEEKS` weeks per cluster (no Dec 2023 context)
6. Load saved model → evaluate

### Test Step 1 · Load 2024 Holdout Data

In [ ]:
TEST_LINK = "https://drive.google.com/file/d/1USTg4Dj60ktjZRMqCNU44JtpfXQ9CWp9/view?usp=sharing"  # 2024

test_raw_df = pd.read_csv(to_csv_url(TEST_LINK))
print(f"2024 raw shape: {test_raw_df.shape}")


### Test Step 2 · Clean and Prepare 2024 Data

In [ ]:
test_df = clean_df(test_raw_df, 'TEST-2024', '2024-01-01', '2024-12-31 23:59:59')
test_df = add_time_features(test_df)
print(f"2024 after cleaning + time features: {test_df.shape}")


### Test Step 3 · Weekly Aggregation Using Saved Cluster Set

In [ ]:
TRAIN_CLUSTERS = joblib.load('lstm_train_clusters.pkl')

test_weekly = build_weekly_agg(test_df, 'TEST-2024', all_clusters=TRAIN_CLUSTERS)
print(test_weekly.head())


### Test Step 4 · Scale 2024 Using Saved Scaler

The scaler is **loaded from disk** — it was fitted on 2021–2023 and is not refitted here.

In [ ]:
scaler = joblib.load('lstm_scaler.pkl')
N_PAST_WEEKS = joblib.load('lstm_n_past_weeks.pkl')

test_weekly['scaled_count'] = scaler.transform(test_weekly[['crime_count']])

print(f"2024 scaled. scale_min={scaler.data_min_[0]:.1f}, scale_max={scaler.data_max_[0]:.1f} (from training)")
print(f"N_PAST_WEEKS = {N_PAST_WEEKS}")


### Test Step 5 · Build 2024 Sequences

First `N_PAST_WEEKS` weeks per cluster are dropped — they would need December 2023 context which doesn't exist in the 2024 file. Consistent with the v3 NaN-drop approach.

In [ ]:
all_X_test, all_y_test = [], []
clusters_with_sequences = []

for cluster in TRAIN_CLUSTERS:
    cluster_scaled = test_weekly[
        test_weekly['NEIGHBORHOOD_CLUSTER'] == cluster
    ]['scaled_count'].values

    # Need at least N_PAST_WEEKS + 1 rows to form one sequence
    # First N_PAST_WEEKS rows dropped because no Dec 2023 context
    if len(cluster_scaled) >= N_PAST_WEEKS + 1:
        X_c, y_c = create_sequences(cluster_scaled, N_PAST_WEEKS)
        all_X_test.append(X_c)
        all_y_test.append(y_c)
        clusters_with_sequences.append(cluster)

X_test_lstm = np.concatenate(all_X_test, axis=0)
y_test_lstm = np.concatenate(all_y_test, axis=0)

# Reshape for LSTM: [samples, timesteps, features]
X_test_lstm = X_test_lstm.reshape((X_test_lstm.shape[0], N_PAST_WEEKS, 1))

print(f"X_test_lstm shape: {X_test_lstm.shape}")
print(f"y_test_lstm shape: {y_test_lstm.shape}")
print(f"Clusters with sequences: {len(clusters_with_sequences)}")
print(f"Note: First {N_PAST_WEEKS} weeks per cluster dropped (no Dec 2023 context)")


### Test Step 6 · Load Saved Model and Evaluate on 2024

In [ ]:
model = tf.keras.models.load_model('lstm_model.keras')
print("Model loaded from lstm_model.keras")

# Predict
y_pred_scaled = model.predict(X_test_lstm)

# Inverse transform back to original crime count scale
y_pred_orig = scaler.inverse_transform(y_pred_scaled)
y_test_orig = scaler.inverse_transform(y_test_lstm.reshape(-1, 1))

print("\n" + "="*50)
print("LSTM — 2024 Holdout Results")
print("="*50)
mae  = mean_absolute_error(y_test_orig, y_pred_orig)
mse  = mean_squared_error(y_test_orig, y_pred_orig)
rmse = np.sqrt(mse)
r2   = r2_score(y_test_orig, y_pred_orig)

print(f"MAE  : {mae:.4f} crimes")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f} crimes")
print(f"R²   : {r2:.4f}")


### Test Step 7 · Visualisations

In [ ]:
# --- Plot 1: Forecast vs Actual for first cluster ---
first_cluster = clusters_with_sequences[0]
n_seq = len(all_y_test[0])   # sequences for first cluster

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('LSTM — 2024 Holdout Evaluation (Trained on 2021-2023 Only)',
             fontsize=13, fontweight='bold')

axes[0].plot(y_test_orig[:n_seq],  label='Actual',    color='steelblue')
axes[0].plot(y_pred_orig[:n_seq],  label='Predicted', color='tomato', linestyle='--')
axes[0].set_title(f'Forecast vs Actual — {first_cluster}')
axes[0].set_xlabel('Weeks in 2024 Test Set')
axes[0].set_ylabel('Crime Count')
axes[0].legend()

# --- Plot 2: Residuals ---
residuals = y_test_orig.flatten() - y_pred_orig.flatten()
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Distribution (Actual − Predicted)')
axes[1].set_xlabel('Residual (crimes)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# --- Plot 3: Scatter actual vs predicted ---
plt.figure(figsize=(6, 6))
plt.scatter(y_test_orig, y_pred_orig, alpha=0.3, s=10, color='steelblue')
max_val = max(y_test_orig.max(), y_pred_orig.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual Crime Count')
plt.ylabel('Predicted Crime Count')
plt.title(f'Actual vs Predicted — 2024 Holdout (R² = {r2:.4f})')
plt.legend()
plt.tight_layout()
plt.show()
